# 02_02：依 PERMNO 人工審核清理 comment–ticker–PERMNO

讀取 pipeline registry 的新版 02 與同 run 的 `top300_permno_review.csv`。僅排除 del=1 的 PERMNO，del=0 及未審核 PERMNO 保留；不再使用舊 ticker 審核表或固定舊來源。

核對審核表的來源 SHA-256 與每個 PERMNO 的 times。完整保留來源欄位、日期、列順序與 PERMNO，不重新配對。每次建立新的輸出目錄，原始資料與人工 del 不改動。unique comments 仍以 `(source_file, source_row)` 識別。


In [1]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import csv, json, hashlib, sqlite3, time
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
import pandas as pd
from IPython.display import display

ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'Data/process').is_dir())
MATCH_RUN = None  # 可指定新版 02 run；預設使用成功 registry。
if MATCH_RUN is None:
    pointer = json.loads((ROOT/'Data/process/pipeline_runs.json').read_text(encoding='utf-8')).get('matching_manifest')
    if not pointer: raise FileNotFoundError('請先成功執行新版 02。')
    manifest = Path(pointer)
    if not manifest.is_absolute(): manifest = ROOT/manifest
    MATCH_RUN = manifest.parent
else:
    MATCH_RUN = Path(MATCH_RUN)
    if not MATCH_RUN.is_absolute(): MATCH_RUN = ROOT/MATCH_RUN
matching_info = json.loads((MATCH_RUN/'run_summary.json').read_text(encoding='utf-8'))
assert matching_info['status']=='complete' and matching_info.get('schema_version',0)>=3
SOURCE = MATCH_RUN/'comment_ticker_long.parquet'
assert 'permno' in pq.read_schema(SOURCE).names, '來源缺少已按留言日期配對的 PERMNO。'
REVIEW = ROOT/'Data/process/02_01_permno_frequency_review'/MATCH_RUN.name/'top300_permno_review.csv'
review_bytes = REVIEW.read_bytes()
review = pd.read_csv(REVIEW,dtype=str,keep_default_na=False,encoding='utf-8-sig')
assert review.columns.tolist()==['permno','ticker','times','del']
assert review.permno.str.fullmatch(r'[1-9][0-9]*').all() and review.permno.is_unique
assert review.times.str.fullmatch(r'[1-9][0-9]*').all()
assert review['del'].isin(['0','1']).all(), 'del 僅接受 0 或 1。'
excluded = set(map(int,review.loc[review['del'].eq('1'),'permno']))
OUT=ROOT/'Data/process/02_02_filtered_comment_ticker'/datetime.now(timezone.utc).strftime('%Y-%m-%d_%H-%M-%S_UTC')
OUT.mkdir(parents=True,exist_ok=False)
DEST=OUT/'comment_ticker_cleaned.parquet'
TEMP=DEST.with_suffix('.parquet.partial')
source_stat=(SOURCE.stat().st_size,SOURCE.stat().st_mtime_ns)
def sha256_file(path):
    digest=hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda:stream.read(4*1024*1024),b''):digest.update(block)
    return digest.hexdigest()
source_hash=sha256_file(SOURCE)
review_meta_path=REVIEW.parent/'run_summary.json'
if review_meta_path.exists():
    review_meta=json.loads(review_meta_path.read_text(encoding='utf-8'))
    assert Path(review_meta['source']).resolve()==SOURCE.resolve()
    assert review_meta['source_sha256']==source_hash, '人工審核來源與目前 long-format 檔案不一致'
print('來源：',SOURCE)
print('審核表列數：',len(review),'；del=1 PERMNO：',len(excluded))
print('输出：',DEST)


來源： c:\Users\eagle\RedditStock\Data\process\02_ticker_matching\2026-10-13_05-01-55_UTC\comment_ticker_long.parquet
審核表列數： 249 ；del=1 PERMNO： 145
输出： c:\Users\eagle\RedditStock\Data\process\02_02_filtered_comment_ticker\2026-10-13_05-51-49_UTC\comment_ticker_cleaned.parquet


## 分批過濾與診斷
只移除排除集合中的 PERMNO，保留完整欄位。


In [2]:
started=time.perf_counter()
parquet=pq.ParquetFile(SOURCE)
assert {'permno','ticker','year_month','thread_id','comment_id','source_file','source_row'}<=set(parquet.schema_arrow.names)
deny=pa.array(sorted(excluded),type=pa.int64())
db_path=OUT/'_diagnostic_work.sqlite'
db=sqlite3.connect(db_path)
db.execute('PRAGMA temp_store=FILE')
db.execute('PRAGMA cache_size=-32768')
db.execute('CREATE TABLE comments(stage TEXT, file TEXT, row INTEGER, PRIMARY KEY(stage,file,row)) WITHOUT ROWID')
db.execute('CREATE TABLE threads(stage TEXT, thread TEXT, PRIMARY KEY(stage,thread)) WITHOUT ROWID')
stats={stage:{'rows':0,'month_min':None,'month_max':None,'missing_month_rows':0,'tickers':set(),'permnos':set()} for stage in ['before','after']}
observed=Counter()
def update(stage,table):
    s=stats[stage];s['rows']+=table.num_rows
    s['tickers'].update(pc.unique(table.column('ticker')).to_pylist())
    s['permnos'].update(pc.unique(table.column('permno')).to_pylist())
    months=[x for x in pc.unique(table.column('year_month')).to_pylist() if x]
    if months:
        s['month_min']=min(s['month_min'],min(months)) if s['month_min'] else min(months)
        s['month_max']=max(s['month_max'],max(months)) if s['month_max'] else max(months)
    missing=pc.or_(pc.is_null(table.column('year_month')),pc.fill_null(pc.equal(table.column('year_month'),''),False))
    s['missing_month_rows']+=int(pc.sum(pc.cast(missing,pa.int64())).as_py() or 0)
    comments=table.select(['source_file','source_row']).to_pandas().drop_duplicates()
    assert comments.notna().all().all()
    db.executemany('INSERT OR IGNORE INTO comments VALUES (?,?,?)',((stage,f,int(r)) for f,r in comments.itertuples(index=False,name=None)))
    db.executemany('INSERT OR IGNORE INTO threads VALUES (?,?)',((stage,t) for t in pc.unique(table.column('thread_id')).to_pylist() if t))
    return {'row_count':table.num_rows,'month_min':min(months) if months else None,'month_max':max(months) if months else None,
            'unique_comments':len(comments),'unique_threads':len([t for t in pc.unique(table.column('thread_id')).to_pylist() if t]),'unique_authors':'N/A'}
with pq.ParquetWriter(TEMP,parquet.schema_arrow,compression='zstd') as writer:
    for batch_no,batch in enumerate(parquet.iter_batches(batch_size=100_000),1):
        table=pa.Table.from_batches([batch])
        assert table.column('ticker').null_count==0
        assert table.column('permno').null_count==0
        assert pc.all(pc.greater(table.column('permno'),0)).as_py()
        for item in pc.value_counts(table.column('permno')).to_pylist():observed[item['values']]+=item['counts']
        cleaned=table.filter(pc.invert(pc.is_in(table.column('permno'),value_set=deny)))
        assert not any(t in excluded for t in pc.unique(cleaned.column('permno')).to_pylist())
        before_diag=update('before',table);after_diag=update('after',cleaned)
        if cleaned.num_rows:writer.write_table(cleaned,row_group_size=100_000)
        db.commit()
        print({'batch':batch_no,'before':before_diag,'after':after_diag},flush=True)
assert stats['before']['rows']==parquet.metadata.num_rows
assert pq.read_metadata(TEMP).num_rows==stats['after']['rows']
# 比對審核表上的 times，避免同名 run 或來源混用。
for row in review.itertuples(index=False,name=None):
    assert observed[int(row[0])]==int(row[2]), ('審核表 times 與來源不同',row[0])
assert stats['before']['rows']-stats['after']['rows']==sum(observed[t] for t in excluded)
for stage,s in stats.items():
    s['unique_tickers']=len(s.pop('tickers'))
    s['unique_permnos']=len(s.pop('permnos'))
    s['unique_comments']=db.execute('SELECT COUNT(*) FROM comments WHERE stage=?',(stage,)).fetchone()[0]
    s['unique_threads']=db.execute('SELECT COUNT(*) FROM threads WHERE stage=?',(stage,)).fetchone()[0]
    s['unique_authors']=None
db.close()
assert REVIEW.read_bytes()==review_bytes
assert source_stat==(SOURCE.stat().st_size,SOURCE.stat().st_mtime_ns)
TEMP.replace(DEST)
db_path.unlink()
report={'status':'complete','source':str(SOURCE),'source_sha256':source_hash,'review_file':str(REVIEW),
        'review_sha256':hashlib.sha256(review_bytes).hexdigest(),'review_rows':len(review),
        'schema_version':2,'review_unit':'permno','excluded_permnos':sorted(excluded),'excluded_permno_count':len(excluded),
        'before':stats['before'],'after':stats['after'],
        'removed_mentions':stats['before']['rows']-stats['after']['rows'],
        'comments_losing_all_tickers':stats['before']['unique_comments']-stats['after']['unique_comments'],
        'output':str(DEST),'processing_seconds':time.perf_counter()-started}
(OUT/'run_summary.json').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
display(pd.DataFrame(stats).T)
preview=next(pq.ParquetFile(DEST).iter_batches(batch_size=10),None)
if preview is not None:display(preview.to_pandas())
print('完成：',DEST)


{'batch': 1, 'before': {'row_count': 100000, 'month_min': '2012-04', 'month_max': '2015-07', 'unique_comments': 46658, 'unique_threads': 6159, 'unique_authors': 'N/A'}, 'after': {'row_count': 19397, 'month_min': '2012-04', 'month_max': '2015-07', 'unique_comments': 14720, 'unique_threads': 4344, 'unique_authors': 'N/A'}}
{'batch': 2, 'before': {'row_count': 100000, 'month_min': '2015-07', 'month_max': '2015-12', 'unique_comments': 48854, 'unique_threads': 5263, 'unique_authors': 'N/A'}, 'after': {'row_count': 15181, 'month_min': '2015-07', 'month_max': '2015-12', 'unique_comments': 12261, 'unique_threads': 3334, 'unique_authors': 'N/A'}}
{'batch': 3, 'before': {'row_count': 100000, 'month_min': '2015-12', 'month_max': '2016-03', 'unique_comments': 50543, 'unique_threads': 5316, 'unique_authors': 'N/A'}, 'after': {'row_count': 14030, 'month_min': '2015-12', 'month_max': '2016-03', 'unique_comments': 11370, 'unique_threads': 3070, 'unique_authors': 'N/A'}}
{'batch': 4, 'before': {'row_co

,rows,month_min,month_max,missing_month_rows,unique_tickers,unique_permnos,unique_comments,unique_threads,unique_authors
before,10215104,2012-04,2020-08,0,3954,3785,6047095,243370,None
after,1447576,2012-04,2020-08,0,3816,3640,1257880,127966,None


,year_month,thread_id,comment_id,ticker,source_file,source_row,permno,created_date,created_datetime_utc,match_status
0,2012-04,s4jw1,c4b0pvu,JPM,wallstreetbets_comment_part1.csv,1,47896,2012-04-11,2012-04-11 16:46:43+00:00,matched
1,2012-04,s4jw1,c4b1fpf,INTC,wallstreetbets_comment_part1.csv,2,59328,2012-04-11,2012-04-11 17:39:08+00:00,matched
2,2012-04,s4jw1,c4b2egm,GMCR,wallstreetbets_comment_part1.csv,4,79588,2012-04-11,2012-04-11 18:47:11+00:00,matched
3,2012-04,s4jw1,c4b2egm,GOOG,wallstreetbets_comment_part1.csv,4,90319,2012-04-11,2012-04-11 18:47:11+00:00,matched
4,2012-04,s4jw1,c4b2egm,GRPN,wallstreetbets_comment_part1.csv,4,13108,2012-04-11,2012-04-11 18:47:11+00:00,matched
5,2012-04,s4jw1,c4b2egm,LVS,wallstreetbets_comment_part1.csv,4,90505,2012-04-11,2012-04-11 18:47:11+00:00,matched
6,2012-04,s4jw1,c4b389t,BZH,wallstreetbets_comment_part1.csv,5,80193,2012-04-11,2012-04-11 19:44:33+00:00,matched
7,2012-04,s4jw1,c4b389t,CROX,wallstreetbets_comment_part1.csv,5,91090,2012-04-11,2012-04-11 19:44:33+00:00,matched
8,2012-04,s4jw1,c4b3hli,GOOG,wallstreetbets_comment_part1.csv,6,90319,2012-04-11,2012-04-11 20:02:56+00:00,matched
9,2012-04,s4jw1,c4b3hli,JPM,wallstreetbets_comment_part1.csv,6,47896,2012-04-11,2012-04-11 20:02:56+00:00,matched


完成： c:\Users\eagle\RedditStock\Data\process\02_02_filtered_comment_ticker\2026-10-13_05-51-49_UTC\comment_ticker_cleaned.parquet


## 附加統計：清理後 thread／author／comment 的五項分布

使用全部清理後資料，不另外限制月份。五項皆輸出 `N, Mean, Median, std, p90, max`：

1. 每個 thread 的不同可辨識 author 數。
2. 每個 thread 的不同 comment 數。
3. 每個 thread 提及的不同 PERMNO 數。
4. 每個可辨識 author 提及的不同 PERMNO 數。
5. 每個 comment 提及的不同 PERMNO 數。

comment 鍵為 `(source_file, source_row)`，不使用可能重複的 comment_id。相同 PERMNO 的 ticker 別名只算一支股票。author 若未在 long-format 中保留，依來源鍵從同一個 02 run 的 comments Parquet 補回；無法對應時停止，不推測作者。

空白、[deleted]、[removed] author 不算可辨識作者，不合併成一個虛構作者；這些留言仍計入 comment／PERMNO 統計。每個有效 thread 都保留，即使可辨識 author 數為 0。缺少 thread ID 的留言不計入前三項，仍可計入後兩項。N 是該分布的 thread／author／comment 數，std 使用 ddof=1，p90 使用線性內插。

只新增 statistics 時間目錄，不修改清理後資料、人工審核表或舊統計。


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, time
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

ACTIVITY_INPUT = None
ac_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'Data/process').is_dir())
if ACTIVITY_INPUT is not None:
    ac_source = Path(ACTIVITY_INPUT)
    if not ac_source.is_absolute(): ac_source = ac_root/ac_source
elif 'DEST' in globals():
    ac_source = Path(DEST)
else:
    ac_candidates = []
    for manifest in (ac_root/'Data/process/02_02_filtered_comment_ticker').glob('*/run_summary.json'):
        info = json.loads(manifest.read_text(encoding='utf-8'))
        source = manifest.parent/'comment_ticker_cleaned.parquet'
        if info.get('status') == 'complete' and source.is_file(): ac_candidates.append(source)
    if not ac_candidates: raise FileNotFoundError('找不到成功的 02_02 輸出')
    ac_source = max(ac_candidates, key=lambda p:p.parent.name)
ac_manifest = json.loads((ac_source.parent/'run_summary.json').read_text(encoding='utf-8'))
if ac_manifest.get('status') != 'complete': raise ValueError('02_02 尚未成功完成')
ac_schema = pq.read_schema(ac_source).names
if 'permno' not in ac_schema: raise ValueError('來源缺少 PERMNO；請執行新版 02_02')
ac_stat = (ac_source.stat().st_size, ac_source.stat().st_mtime_ns)
ac_started = time.perf_counter()
ac_columns = ['source_file','source_row','thread_id','permno']
if 'author' in ac_schema: ac_columns.append('author')
ac_mentions = pd.read_parquet(ac_source, columns=ac_columns)
if ac_mentions[['source_file','source_row','permno']].isna().any().any(): raise ValueError('缺少來源鍵或 PERMNO')
if not ac_mentions.permno.gt(0).all() or not ac_mentions.permno.mod(1).eq(0).all(): raise ValueError('PERMNO 必須為正整數')
ac_mentions['permno'] = ac_mentions.permno.astype('int64')
ac_mentions['thread_id'] = ac_mentions.thread_id.fillna('').str.strip()
ac_comment_columns = ['source_file','source_row','thread_id'] + (['author'] if 'author' in ac_columns else [])
ac_comments = ac_mentions[ac_comment_columns].drop_duplicates()
if ac_comments.duplicated(['source_file','source_row']).any(): raise ValueError('同一留言的 thread/author 不一致')
ac_author_sources = []
ac_author_input_stats = {}
if 'author' not in ac_comments:
    ac_matching_source = Path(ac_manifest['source'])
    if not ac_matching_source.is_absolute(): ac_matching_source = ac_root/ac_matching_source
    ac_matching_run = ac_matching_source.parent
    ac_parts = sorted(ac_matching_run.glob('*.comments.parquet'))
    if not ac_parts: raise FileNotFoundError('找不到來源 02 comments Parquet，無法補回 author')
    ac_keys = ac_comments[['source_file','source_row']]
    ac_recovered = []
    for part in ac_parts:
        ac_author_input_stats[str(part)] = (part.stat().st_size,part.stat().st_mtime_ns)
        source_file = part.name.removesuffix('.comments.parquet') + '.csv'
        wanted = ac_keys.loc[ac_keys.source_file.eq(source_file)]
        if wanted.empty: continue
        for batch in pq.ParquetFile(part).iter_batches(batch_size=100_000, columns=['source_file','source_row','author']):
            matched = batch.to_pandas().merge(wanted,on=['source_file','source_row'],how='inner',validate='one_to_one')
            if len(matched): ac_recovered.append(matched)
        ac_author_sources.append(str(part))
        print('author recovery:',part.name,flush=True)
    if not ac_recovered: raise ValueError('沒有留言成功取得 author')
    authors = pd.concat(ac_recovered,ignore_index=True)
    ac_comments = ac_comments.merge(authors,on=['source_file','source_row'],how='left',validate='one_to_one',indicator=True)
    if not ac_comments['_merge'].eq('both').all(): raise ValueError('清理後留言無法對應來源 author')
    ac_comments.drop(columns='_merge',inplace=True)
ac_comments['author'] = ac_comments.author.fillna('').str.strip()
ac_comments['known_author'] = ~ac_comments.author.str.lower().isin(['','[deleted]','[removed]'])
ac_mentions = ac_mentions.drop(columns='author',errors='ignore').merge(
    ac_comments[['source_file','source_row','author','known_author']],on=['source_file','source_row'],how='left',validate='many_to_one')
valid_threads = ac_comments.loc[ac_comments.thread_id.ne('')]
ac_thread = valid_threads.groupby('thread_id').size().rename('n_comments').to_frame()
ac_thread['n_authors'] = valid_threads.loc[valid_threads.known_author].groupby('thread_id').author.nunique().reindex(ac_thread.index,fill_value=0)
ac_thread['n_permnos'] = ac_mentions.loc[ac_mentions.thread_id.ne('')].groupby('thread_id').permno.nunique().reindex(ac_thread.index,fill_value=0)
ac_thread = ac_thread.reset_index()[['thread_id','n_authors','n_comments','n_permnos']]
ac_author = ac_mentions.loc[ac_mentions.known_author].groupby('author').permno.nunique().rename('n_permnos').reset_index()
ac_comment = ac_mentions.groupby(['source_file','source_row']).permno.nunique().rename('n_permnos').reset_index()
assert int(ac_thread.n_comments.sum()) == len(valid_threads)
assert len(ac_comment) == len(ac_comments)
assert len(ac_mentions) == pq.read_metadata(ac_source).num_rows == ac_manifest['after']['rows']

def activity_distribution(label, values):
    a = np.asarray(values,dtype=np.int64)
    return {'metric':label, 'N':len(a), 'Mean':float(a.mean()) if len(a) else None,
            'Median':float(np.median(a)) if len(a) else None,
            'std':float(a.std(ddof=1)) if len(a)>1 else None,
            'p90':float(np.percentile(a,90,method='linear')) if len(a) else None,
            'max':int(a.max()) if len(a) else None}

ac_distributions = pd.DataFrame([
    activity_distribution('authors_per_thread',ac_thread.n_authors),
    activity_distribution('comments_per_thread',ac_thread.n_comments),
    activity_distribution('permnos_per_thread',ac_thread.n_permnos),
    activity_distribution('permnos_per_author',ac_author.n_permnos),
    activity_distribution('permnos_per_comment',ac_comment.n_permnos),
])
assert ac_stat == (ac_source.stat().st_size,ac_source.stat().st_mtime_ns)
for part,stat in ac_author_input_stats.items():
    assert stat == (Path(part).stat().st_size,Path(part).stat().st_mtime_ns)
ac_out = ac_source.parent/'statistics'/datetime.now(timezone.utc).strftime('%Y-%m-%d_%H-%M-%S_UTC')
ac_out.mkdir(parents=True,exist_ok=False)
ac_distributions.to_csv(ac_out/'activity_distributions.csv',index=False,encoding='utf-8-sig')
ac_thread.to_parquet(ac_out/'thread_statistics.parquet',index=False,compression='zstd')
ac_author.to_parquet(ac_out/'author_statistics.parquet',index=False,compression='zstd')
ac_comment.to_parquet(ac_out/'comment_statistics.parquet',index=False,compression='zstd')
ac_summary = {'status':'complete','source':str(ac_source),'mention_rows':len(ac_mentions),
    'unique_comments':len(ac_comments),'unique_threads':len(ac_thread),'unique_authors':len(ac_author),
    'comments_missing_thread':int(ac_comments.thread_id.eq('').sum()),
    'comments_unknown_author':int((~ac_comments.known_author).sum()),
    'comment_key':['source_file','source_row'],'permno_count':'distinct PERMNOs, not ticker mentions',
    'author_policy':'Exclude empty/[deleted]/[removed] identities only from author counts; retain their comments.',
    'population':'all cleaned retained mentions across the full period; no additional date filter',
    'std_ddof':1,'p90_method':'linear','author_sources':ac_author_sources,
    'distribution':ac_distributions.to_dict(orient='records'),'runtime_seconds':time.perf_counter()-ac_started}
(ac_out/'activity_summary.json').write_text(json.dumps(ac_summary,ensure_ascii=False,indent=2),encoding='utf-8')
display(ac_distributions)
print('統計輸出：',ac_out)
